| 파라미터             | 의미          | 영향         |
| ---------------- | ----------- | ---------- |
| n_estimators     | 트리 개수       | 많으면 과적합 위험 |
| max_depth        | 트리 깊이       | 클수록 복잡     |
| learning_rate    | 학습률         | 작을수록 안정    |
| subsample        | 샘플 비율       | 과적합 방지     |
| colsample_bytree | 변수 비율       | 과적합 방지     |
| gamma            | split 최소 손실 | 클수록 보수적    |
| reg_alpha        | L1          | sparsity   |
| reg_lambda       | L2          | 안정화        |


In [2]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score

# 데이터
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 모델
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9649122807017544


In [3]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 300],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

model = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}
Best Score: 0.9736263736263737


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    'max_depth': np.arange(3, 10),
    'learning_rate': np.linspace(0.01, 0.3, 20),
    'n_estimators': np.arange(100, 1000, 100),
    'subsample': np.linspace(0.6, 1.0, 5),
    'colsample_bytree': np.linspace(0.6, 1.0, 5),
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 5, 10]
}

random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Params:", random_search.best_params_)

✅ 2️⃣ XGBoost eval_metric (Classification)

| metric    | 의미                     |
| --------- | ---------------------- |
| logloss   | 로지스틱 손실 (기본)           |
| error     | 오분류율                   |
| auc       | ROC AUC                |
| aucpr     | PR AUC                 |
| map       | mean average precision |
| f1        | F1-score               |
| precision | 정밀도                    |
| recall    | 재현율                    |
✅ 3️⃣ Regression eval_metric

| metric   | 의미                  |
| -------- | ------------------- |
| mlogloss | multi-class logloss |
| merror   | multi-class error   |
| auc      | multi-class AUC     |

4️⃣ sklearn scoring에서 쓰는 것들
🔹 Classification
| scoring           | 의미         |
| ----------------- | ---------- |
| accuracy          | 정확도        |
| precision         | 정밀도        |
| recall            | 재현율        |
| f1                | F1         |
| roc_auc           | ROC AUC    |
| roc_auc_ovr       | 다중 AUC     |
| average_precision | PR AUC     |
| neg_log_loss      | 로그손실       |
| balanced_accuracy | 불균형 보정 정확도 |

🔹 Regression
| scoring                            | 의미   |
| ---------------------------------- | ---- |
| neg_mean_squared_error             | MSE  |
| neg_root_mean_squared_error        | RMSE |
| neg_mean_absolute_error            | MAE  |
| r2                                 | R²   |
| neg_mean_absolute_percentage_error | MAPE |

🔹 Binary 분류

eval_metric = "auc"

scoring = "roc_auc"

🔹 불균형 데이터

eval_metric = "aucpr"

scoring = "average_precision"

🔹 회귀

eval_metric = "rmse"

scoring = "neg_root_mean_squared_error"


In [ ]:
# XGBoost 내부 metric
def custom_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    return "custom_error", float(((y_pred > 0.5) != y_true).mean())

model = XGBClassifier(eval_metric=custom_eval)
# sklearn scoring custom
from sklearn.metrics import make_scorer, f1_score

custom_scorer = make_scorer(f1_score)

GridSearchCV(model, param_grid, scoring=custom_scorer)

# 다중 metric 동시에 쓰기 (sklearn)
scoring = {
    "AUC": "roc_auc",
    "F1": "f1",
    "Accuracy": "accuracy"
}

grid = GridSearchCV(
    model,
    param_grid,
    scoring=scoring,
    refit="AUC",
    cv=5
)